# Ejercicio: asistente de documentos con NVIDIA NIM y Gradio

**Tiempo:** 40–45 minutos  
**Grupos:** 3–4 personas

## Contexto

En los notebooks anteriores construimos chatbots que responden preguntas
generales. En este ejercicio construiremos un asistente que responda preguntas
sobre un documento específico: **“Attention Is All You Need”**
(Vaswani et al., 2017).

Inyectar un documento completo en el contexto permite construir un primer
prototipo de *document question answering*. Esto ayuda a entender la motivación
de **RAG**, aunque todavía no es RAG: aquí no existe recuperación de fragmentos.

## Objetivo

Construir una aplicación Gradio donde el usuario pueda:

1. Cargar o leer el PDF del paper.
2. Hacer preguntas sobre su contenido.
3. Obtener respuestas basadas exclusivamente en el documento.

## Lo que van a aprender

- Extracción de texto de PDF con `pypdf`.
- Uso de contexto externo en un `system` prompt.
- Consumo de NVIDIA NIM mediante una API compatible con OpenAI.
- Streaming e historial con Gradio.
- Limitaciones de este enfoque y motivación de RAG.

> **Estado de esta entrega (Juan David Sierra Orozco, SI3003 2026-2).**
> Todo el código está resuelto y ejecutado. Las celdas que no necesitan la API (extracción del PDF, construcción del prompt, interfaz Gradio, medición de contexto, recuperación por palabras) tienen salidas reales. Las celdas que llaman a NVIDIA NIM quedan **bloqueadas** porque en este entorno no existe la variable `NVIDIA_API_KEY`: el notebook lo detecta y lo informa, no simula ninguna respuesta del modelo. Con la clave en un `.env` basta con ejecutar de nuevo.

## Paso 0: instalación y configuración

### 0.1 Instala las dependencias

- `pypdf`: extracción del texto del PDF.
- `gradio`: interfaz web.
- `openai`: cliente compatible con NVIDIA NIM.
- `python-dotenv`: lectura de variables de entorno.

```bash
pip install -q --upgrade pypdf gradio openai python-dotenv
```

In [1]:
import sys, importlib.metadata as md
for p in ["pypdf", "gradio", "openai", "python-dotenv"]:
    print(p, md.version(p))
print("python", sys.version.split()[0])

pypdf 6.20.1
gradio 6.30.0
openai 3.31.0
python-dotenv 1.2.4
python 3.12.14


### 0.2 Descarga el paper

Descarga el PDF abierto desde:

```text
https://arxiv.org/pdf/1706.03762
```

Guárdalo junto al notebook como `attention_is_all_you_need.pdf`.

### 0.3 Configura tus credenciales

Crea un archivo `.env` con tu API key de NVIDIA:

```text
NVIDIA_API_KEY="tu_key_aqui"
```

Puedes crearla en [NVIDIA Build](https://build.nvidia.com/settings/api-keys).

In [2]:
from pathlib import Path
PDF_PATH = Path("attention_is_all_you_need.pdf")   # copia local de arxiv.org/pdf/1706.03762
print(PDF_PATH, "existe:", PDF_PATH.exists(), f"({PDF_PATH.stat().st_size/1e6:.2f} MB)")

attention_is_all_you_need.pdf existe: True (2.22 MB)


## Paso 1: extracción de texto del PDF

Lee el PDF y extrae su contenido como texto plano.

**Instrucciones:**

1. Importa `PdfReader` desde `pypdf`.
2. Crea `extract_text_from_pdf(pdf_path)`.
3. Abre el PDF e itera sobre `reader.pages`.
4. Concatena el resultado de `page.extract_text()`; contempla páginas sin texto.
5. Retorna el texto completo.
6. Prueba la función e imprime los primeros 500 caracteres.

> **Pista:** usa `page.extract_text() or ""` para evitar valores `None`.

In [3]:
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):
    """Texto plano del PDF. Maneja paginas sin texto con `or ""`."""
    reader = PdfReader(pdf_path)
    paginas = []
    for page in reader.pages:
        paginas.append(page.extract_text() or "")
    return "\n".join(paginas)

texto = extract_text_from_pdf(PDF_PATH)
print("Caracteres:", len(texto))
print("--- primeros 500 caracteres ---")
print(texto[:500])

Caracteres: 39599
--- primeros 500 caracteres ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


## Paso 2: inicialización del cliente de NVIDIA NIM

**Instrucciones:**

1. Importa `os`, `load_dotenv` y `OpenAI`.
2. Carga las variables con `load_dotenv()`.
3. Verifica que `NVIDIA_API_KEY` esté disponible sin imprimirla.
4. Inicializa `OpenAI` con:
   - `base_url="https://integrate.api.nvidia.com/v1"`
   - `api_key=os.getenv("NVIDIA_API_KEY")`
5. Define:

```python
MODELO = "nvidia/nemotron-3-super-120b-a12b"
```

In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
MODELO = "nvidia/nemotron-3-super-120b-a12b"
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
HAY_CLAVE = bool(NVIDIA_API_KEY)
print("NVIDIA_API_KEY disponible:", HAY_CLAVE)   # nunca se imprime el valor

client = None
if HAY_CLAVE:
    client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=NVIDIA_API_KEY)
else:
    print("BLOQUEADO: no hay NVIDIA_API_KEY ni en el entorno ni en un .env. "
          "Las llamadas al modelo no se ejecutan; el resto del notebook si.")

NVIDIA_API_KEY disponible: False
BLOQUEADO: no hay NVIDIA_API_KEY ni en el entorno ni en un .env. Las llamadas al modelo no se ejecutan; el resto del notebook si.


## Paso 3: función de chat con el documento

Construye un `system` prompt que incluya el texto del paper y obligue al modelo
a fundamentar sus respuestas en él.

**Instrucciones:**

1. Crea `build_system_prompt(document_text)` para:
   - definir al asistente como experto en el paper;
   - incluir el documento entre delimitadores claros;
   - exigir respuestas basadas únicamente en el documento;
   - indicar que responda “No encuentro esa información en el documento” cuando corresponda.

2. Crea `history_to_messages(history)` para convertir el historial de Gradio a
   mensajes con roles `user` y `assistant`.

3. Crea `chat_con_documento(message, history, document_text)` para:
   - iniciar `messages` con el `system` prompt;
   - agregar el historial y el mensaje actual;
   - llamar `client.chat.completions.create(..., stream=True)`;
   - acumular `chunk.choices[0].delta.content`;
   - usar `yield` para entregar la respuesta progresivamente.

> **Pista:** utiliza `temperature=1.0` y `top_p=0.95`, valores recomendados para
> Nemotron 3 Super.

In [5]:
NO_ENCUENTRO = "No encuentro esa información en el documento"

def build_system_prompt(document_text):
    return f"""Eres un asistente experto en el paper "Attention Is All You Need" (Vaswani et al., 2017).
Responde en el mismo idioma de la pregunta.

REGLAS:
1. Responde ÚNICAMENTE con información contenida en el documento delimitado abajo.
2. No uses conocimiento previo ni externo, aunque sepas la respuesta.
3. Si la respuesta no está en el documento, responde exactamente: "{NO_ENCUENTRO}".
4. Sé conciso y, cuando sea posible, indica la sección o la página donde está la evidencia.

<documento>
{document_text}
</documento>"""

def history_to_messages(history):
    """Historial de Gradio -> mensajes user/assistant.
    Soporta el formato nuevo (lista de dicts con role/content) y el viejo (pares)."""
    messages = []
    for turn in history:
        if isinstance(turn, dict):
            content = turn.get("content", "")
            if isinstance(content, list):   # Gradio 6 puede entregar bloques de contenido
                content = "".join(b.get("text", "") for b in content if isinstance(b, dict))
            messages.append({"role": turn["role"], "content": content})
        else:
            user, assistant = turn
            messages.append({"role": "user", "content": user})
            if assistant:
                messages.append({"role": "assistant", "content": assistant})
    return messages

def chat_con_documento(message, history, document_text):
    if client is None:
        yield "BLOQUEADO: falta NVIDIA_API_KEY. Define la variable en un archivo .env y reinicia."
        return
    messages = [{"role": "system", "content": build_system_prompt(document_text)}]
    messages += history_to_messages(history)
    messages.append({"role": "user", "content": message})
    stream = client.chat.completions.create(
        model=MODELO, messages=messages, temperature=1.0, top_p=0.95,
        max_tokens=2048, stream=True,
    )
    respuesta = ""
    for chunk in stream:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta.content
        if delta:
            respuesta += delta
            yield respuesta

# Pruebas que no necesitan la API
sp = build_system_prompt("TEXTO_DE_PRUEBA")
print(sp[-120:])
h = [{"role": "user", "content": "hola"}, {"role": "assistant", "content": "hola, ¿qué quieres saber?"}]
print(history_to_messages(h))
print(history_to_messages([("p1", "r1")]))

 y, cuando sea posible, indica la sección o la página donde está la evidencia.

<documento>
TEXTO_DE_PRUEBA
</documento>
[{'role': 'user', 'content': 'hola'}, {'role': 'assistant', 'content': 'hola, ¿qué quieres saber?'}]
[{'role': 'user', 'content': 'p1'}, {'role': 'assistant', 'content': 'r1'}]


## Paso 4: interfaz Gradio

Construye la interfaz con `gr.ChatInterface`.

**Instrucciones:**

1. Extrae el texto del PDF con la función del paso 1.
2. Muestra cuántas páginas y caracteres se extrajeron.
3. Configura `gr.ChatInterface` con:
   - `fn=chat_con_documento`;
   - un título y una descripción;
   - un `gr.Textbox` oculto dentro de `additional_inputs` para enviar el documento;
   - al menos tres preguntas de ejemplo.

```python
gr.Textbox(value=document_text, visible=False)
```

4. Lanza la interfaz:

```python
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)
```

In [6]:
import gradio as gr
print("gradio", gr.__version__)

document_text = extract_text_from_pdf(PDF_PATH)
n_paginas = len(PdfReader(PDF_PATH).pages)
print(f"Paginas: {n_paginas} | Caracteres: {len(document_text):,}")

demo = gr.ChatInterface(
    fn=chat_con_documento,
    title="Asistente del paper: Attention Is All You Need",
    description="Pregunta sobre el paper de Vaswani et al. (2017). Responde solo con base en el documento.",
    additional_inputs=[gr.Textbox(value=document_text, visible=False)],
    examples=[
        ["¿Cuál es la arquitectura principal propuesta en el paper?", document_text],
        ["¿Cuántas capas tiene el encoder del modelo base?", document_text],
        ["¿Qué es GPT-4?", document_text],
    ],
)

# En el curso se lanza con: demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)
# Aqui se levanta sin bloquear el notebook, se comprueba que responde y se cierra.
demo.launch(server_name="127.0.0.1", server_port=8080, prevent_thread_lock=True, show_error=True)
import urllib.request
print("HTTP", urllib.request.urlopen("http://127.0.0.1:8080").status, "-> la interfaz arranca")
demo.close()

/Users/juandavidsierraorozco/.venvs/llm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


gradio 6.30.0


Paginas: 15 | Caracteres: 39,599


* Running on local URL:  http://127.0.0.1:8080
* To create a public link, set `share=True` in `launch()`.


HTTP 200 -> la interfaz arranca


Closing server running on port: 8080


## Paso 5: Prueba y reflexión
 
Una vez que la interfaz esté funcionando, prueba estas preguntas:
 
1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*
2. *"¿Qué es el mecanismo de atención?"*
3. *"¿Cuántas capas tiene el encoder del modelo base?"*
4. *"¿Quiénes son los autores del paper?"*
5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*
 
Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*
 
Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.
¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?
 

In [7]:
preguntas = [
    "¿Cuál es la arquitectura principal propuesta en el paper?",
    "¿Qué es el mecanismo de atención?",
    "¿Cuántas capas tiene el encoder del modelo base?",
    "¿Quiénes son los autores del paper?",
    "¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?",
    "¿Qué es GPT-4?",   # pregunta trampa
]

def preguntar(p):
    return "".join(list(chat_con_documento(p, [], document_text))[-1:])

if HAY_CLAVE:
    for p in preguntas:
        print("P:", p); print("R:", preguntar(p)); print("-" * 60)
else:
    print("BLOQUEADO: sin NVIDIA_API_KEY no se puede consultar a Nemotron.")
    print("Las 6 preguntas quedan listas en la lista `preguntas`; al definir la clave este bloque las ejecuta.")

BLOQUEADO: sin NVIDIA_API_KEY no se puede consultar a Nemotron.
Las 6 preguntas quedan listas en la lista `preguntas`; al definir la clave este bloque las ejecuta.


### Verificación de las respuestas esperadas contra el texto del PDF

Como el modelo no se pudo consultar, se comprueba directamente en el texto extraído qué contiene el paper sobre cada pregunta. Esto no reemplaza la respuesta del modelo: sirve de **criterio de corrección** para cuando se ejecute con la clave (la respuesta correcta del modelo debe coincidir con esta evidencia, y para GPT-4 debe ser “No encuentro esa información en el documento”).

In [8]:
import re
def evidencia(patron, ancho=110, maximo=2):
    out = []
    for m in re.finditer(patron, document_text):
        a, b = max(0, m.start()-ancho), min(len(document_text), m.end()+ancho)
        out.append(document_text[a:b].replace("\n", " "))
        if len(out) >= maximo: break
    return out or ["(sin coincidencias)"]

for titulo, patron in [
    ("Arquitectura principal", r"propose a new simple network architecture"),
    ("Capas del encoder", r"stack of N\s*=\s*6"),
    ("Autores", r"Ashish Vaswani"),
    ("WMT14 EN-DE", r"28\.4 BLEU"),
    ("Pregunta trampa: GPT-4", r"GPT-?4"),
]:
    print(f"## {titulo}")
    for e in evidencia(patron): print("  ...", e, "...")

## Arquitectura principal
  ... a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. ...
## Capas del encoder
  ...  right halves of Figure 1, respectively. 3.1 Encoder and Decoder Stacks Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the  ...
  ... s the embedding layers, produce outputs of dimension dmodel = 512. Decoder: The decoder is also composed of a stack of N = 6identical layers. In addition to the two sub-layers in each encoder layer, the decoder inserts a third sub-lay ...
## Autores
  ... tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ G

**Lectura de la verificación.** El paper sí contiene la evidencia para las cinco primeras preguntas (la arquitectura Transformer, `N = 6` capas idénticas en el encoder, los autores y el resultado de 28.4 BLEU en WMT 2014 inglés a alemán). Para **GPT-4** no hay ninguna coincidencia, lo cual es esperable: el paper es de 2017. Por eso es una buena pregunta trampa: un modelo que respete el `system` prompt debe responder “No encuentro esa información en el documento”; si explica qué es GPT-4, está usando conocimiento previo y desobedeciendo la instrucción.

No se afirma cómo respondió Nemotron porque no se pudo ejecutar.

## Paso 6 (adicional): mejora el sistema

Si terminan antes, implementen una mejora:

**A) Medición del contexto**  
Reporta la cantidad de caracteres o estima los tokens del documento. Explica
por qué caracteres y tokens no son equivalentes.

**B) Subida dinámica de PDF**  
Agrega `gr.File` para cargar cualquier PDF y extrae el texto dentro del flujo de
la aplicación.

**C) Evidencia y páginas**  
Conserva separadores de página al extraer el PDF y exige que cada respuesta cite
la página que contiene la evidencia.

**D) Primer paso hacia RAG**  
Divide el documento en fragmentos y selecciona los más relevantes mediante una
búsqueda simple por palabras antes de llamar al modelo.

## Paso 6A: medición del contexto

In [9]:
n_chars = len(document_text)
n_palabras = len(document_text.split())
# Heuristica comun: ~4 caracteres por token en ingles; solo es una estimacion.
tokens_est_chars = n_chars / 4
tokens_est_palabras = n_palabras / 0.75
print(f"Caracteres: {n_chars:,}")
print(f"Palabras:   {n_palabras:,}")
print(f"Tokens estimados (chars/4):        ~{tokens_est_chars:,.0f}")
print(f"Tokens estimados (palabras/0.75):  ~{tokens_est_palabras:,.0f}")

Caracteres: 39,599
Palabras:   6,111
Tokens estimados (chars/4):        ~9,900
Tokens estimados (palabras/0.75):  ~8,148


Caracteres y tokens **no son equivalentes**: el modelo no lee caracteres sino tokens, que son fragmentos de subpalabras producidos por un tokenizador (BPE o similar). Una palabra común puede ser un solo token, una palabra rara o una fórmula se parte en varios, y el texto extraído de un PDF con ecuaciones, subíndices y guiones de corte genera muchos tokens por carácter. Además el costo y el límite de la ventana de contexto se miden en tokens. Las cifras de arriba son estimaciones (sin el tokenizador real de Nemotron), suficientes para saber que el paper cabe holgadamente en una ventana grande, pero no para facturar.

## Paso 6C: evidencia y páginas

In [10]:
def extract_text_with_pages(pdf_path):
    reader = PdfReader(pdf_path)
    return "\n".join(f"[Página {i}]\n{(p.extract_text() or '')}" for i, p in enumerate(reader.pages, 1))

def build_system_prompt_paginas(document_text):
    base = build_system_prompt(document_text)
    return base.replace("4. Sé conciso y, cuando sea posible, indica la sección o la página donde está la evidencia.",
                        "4. Sé conciso y CITA SIEMPRE la página que contiene la evidencia con el formato (Página N). "
                        "Los marcadores [Página N] del documento indican el número de cada página.")

doc_paginas = extract_text_with_pages(PDF_PATH)
print(doc_paginas[:120].replace("\n", " | "))
print("Marcadores de página encontrados:", len(re.findall(r"\[Página \d+\]", doc_paginas)))
print(build_system_prompt_paginas("X")[300:560])

[Página 1] | Provided proper attribution is provided, Google hereby grants permission to | reproduce the tables and figures 
Marcadores de página encontrados: 15
i la respuesta no está en el documento, responde exactamente: "No encuentro esa información en el documento".
4. Sé conciso y CITA SIEMPRE la página que contiene la evidencia con el formato (Página N). Los marcadores [Página N] del documento indican el número 


## Paso 6D: primer paso hacia RAG (fragmentos y búsqueda por palabras)

In [11]:
STOP = set("el la los las de del que qué cuál cuáles cuántas cuántos es en un una y a por con para se al lo como quién quiénes the of in and to is a".split())

def trocear(texto_paginado, tam=1200, solape=200):
    """Fragmentos de ~tam caracteres con solape, recordando la pagina de origen."""
    chunks = []
    for bloque in re.split(r"(?=\[Página \d+\])", texto_paginado):
        m = re.match(r"\[Página (\d+)\]", bloque)
        if not m: continue
        pag = int(m.group(1)); cuerpo = bloque[m.end():]
        i = 0
        while i < len(cuerpo):
            chunks.append({"pagina": pag, "texto": cuerpo[i:i+tam]})
            i += tam - solape
    return chunks

def tokens(s):
    return [w for w in re.findall(r"\w+", s.lower()) if w not in STOP and len(w) > 2]

def top_fragmentos(pregunta, chunks, k=3):
    q = set(tokens(pregunta))
    puntuados = [(len(q & set(tokens(c["texto"]))), c) for c in chunks]
    return [c for s, c in sorted(puntuados, key=lambda x: -x[0])[:k] if s > 0]

chunks = trocear(doc_paginas)
print("Fragmentos:", len(chunks))

# Las preguntas del paso 5 en ingles y espanol: la busqueda es lexica, asi que solo ve palabras compartidas.
for p in ["How many layers does the encoder have?", "What is the BLEU score on WMT 2014 English-to-German?", "multi-head attention heads", "GPT-4"]:
    print(f"\nP: {p}")
    for c in top_fragmentos(p, chunks):
        print(f"  pag {c['pagina']}: {c['texto'][:90].replace(chr(10), ' ')}...")
    if not top_fragmentos(p, chunks): print("  (ningun fragmento relevante)")

def chat_rag_simple(message, history):
    frags = top_fragmentos(message, chunks, k=4)
    contexto = "\n\n".join(f"[Página {c['pagina']}] {c['texto']}" for c in frags) or "(sin fragmentos relevantes)"
    return build_system_prompt_paginas(contexto)   # se enviaria como system prompt en lugar del documento completo
ctx = chat_rag_simple("How many layers does the encoder have?", [])
print(f"\nContexto con RAG simple: {len(ctx):,} caracteres vs documento completo: {len(build_system_prompt(document_text)):,}")

Fragmentos: 47

P: How many layers does the encoder have?
  pag 6: s of self-attention layers to the recurrent and convolu- tional layers commonly used for m...
  pag 6:  Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential oper...
  pag 13:  Attention Visualizations Input-Input Layer5 It is in this spirit that a majority of Ameri...

P: What is the BLEU score on WMT 2014 English-to-German?
  pag 1: periments on two machine translation tasks show these models to be superior in quality whi...
  pag 8:  Label Smoothing During training, we employed label smoothing of value ϵls = 0.1 [36]. Thi...
  pag 7: by a factor of k. Separable convolutions [ 6], however, decrease the complexity considerab...

P: multi-head attention heads
  pag 5:  output values. These are concatenated and once again projected, resulting in the final va...
  pag 1: periments on two machine translation tasks show these models to be superior in quality whi...
  pag 1: aspect of this work

**Lo que muestra la búsqueda.** El contexto baja de unos 40 mil caracteres (40,127) a unos 5 mil (5,067), pero la recuperación por palabras es frágil, y la salida lo prueba: para "How many layers does the encoder have?" los fragmentos recuperados son de las páginas 6 y 13 y **no** el de la sección 3.1 (página 3), que es el que contiene "N = 6". Para la pregunta del BLEU tampoco aparece la página del resultado principal en primer lugar. Solo encuentra lo que comparte vocabulario literal y las palabras genéricas ("layers", "encoder") aparecen por todas partes. Además, las preguntas en español no recuperan nada del paper en inglés. Esa es la motivación de los embeddings, que recuperan por significado, y de reordenar los resultados.

## Reflexión final

Discutan en grupo:

1. ¿Cuál es la limitación de enviar siempre el documento completo?
2. ¿Por qué este ejercicio todavía no implementa RAG?
3. ¿Cómo comprobarían que la respuesta proviene del documento y no del
   conocimiento previo del modelo?
4. ¿Qué riesgos aparecen al insertar directamente documentos de usuarios en el
   prompt?

## Recursos

- [pypdf](https://pypdf.readthedocs.io)
- [Gradio](https://www.gradio.app/docs)
- [NVIDIA Build](https://build.nvidia.com)
- [Nemotron 3 Super](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b)
- [Paper original](https://arxiv.org/abs/1706.03762)

### Respuestas de reflexión

1. **Limitación de enviar siempre el documento completo.** El costo y la latencia crecen con cada turno porque el documento viaja entero en cada llamada; el contexto es finito, así que un documento largo o varios documentos no caben; y un contexto enorme diluye la atención del modelo sobre la parte relevante (el problema de “perdido en el medio”).
2. **Por qué todavía no es RAG.** RAG necesita una etapa de *recuperación*: indexar el corpus en fragmentos, buscar los más relevantes para cada pregunta e inyectar solo esos. Aquí no se recupera nada, se inyecta todo. El Paso 6D es apenas un esbozo léxico de esa etapa.
3. **Cómo verificar que la respuesta viene del documento.** Pedir cita de página y comprobarla contra el texto (Paso 6C); probar preguntas trampa cuya respuesta no está (GPT-4) y exigir la frase de “no encuentro”; y hacer preguntas cuya respuesta en el documento contradice el conocimiento común, para ver cuál prevalece.
4. **Riesgos de insertar documentos de usuarios en el prompt.** *Prompt injection*: un documento puede contener instrucciones (“ignora las reglas anteriores y…”) que el modelo trate como órdenes; fuga de datos entre usuarios si el contexto se comparte o se registra; costo descontrolado con archivos enormes; y contenido malicioso o engañoso presentado como si fuera una fuente confiable.